In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from collections import Counter
import time
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Dataset Class
class BrakeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# z-score standardize
def zscore_normalize(train, test):
    # Flatten into two dimensions for easy statistics
    train_2d = train.reshape(-1, train.shape[-1])
    mean = np.mean(train_2d, axis=0)
    std = np.std(train_2d, axis=0)
    train_norm = (train - mean) / (std + 1e-8)
    test_norm = (test - mean) / (std + 1e-8)
    return train_norm, test_norm

#  Remap labels to two categories: working (0) and non-working (1) 
def remap_labels(y):
    return np.array([0 if label == 0 else 1 for label in y])


# Class balance check
def print_class_distribution(labels, name):
    counter = Counter(labels)
    print(f"{name} class distribution: {dict(counter)}")

# FCN Model
class FCN(nn.Module):
    def __init__(self, input_channels, num_classes, dropout_p=0.3):  
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x
    
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    model = model.to(device)
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

    return model, val_accuracy

# Test
def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())
    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy


if __name__ == "__main__":
    # file path
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    # Load data
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    # ==== Remap labels to 2 categories ====
    y_train = remap_labels(y_train)
    y_test = remap_labels(y_test)

    print_class_distribution(y_train, "Train")
    print_class_distribution(y_test, "Test")


    # Z-Score Standardize
    x_train, x_test = zscore_normalize(x_train, x_test)

    # transfer into (batch, channel, timestep)
    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    # Split into training/validation sets
    dataset = BrakeDataset(x_train, y_train)
    test_dataset = BrakeDataset(x_test, y_test)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    # DataLoader
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_channels = x_train.shape[1]
    num_classes = 2
    model = FCN(input_channels=input_channels, num_classes=num_classes, dropout_p=0.3)


    # Compute class weights
    
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print("Class weights:", class_weights)


    # Loss function and Optimizer
    # criterion = nn.CrossEntropyLoss()
    # Weighted loss function definition
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)  

    # Train
    epochs = 30
    model, best_val_acc = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        epochs=epochs, device=device
    )

    # Test
    test_accuracy = test_model(model, test_loader, device)
    print(f"Test Accuracy: {test_accuracy:.4f}")

Train class distribution: {1: 2255, 0: 1798}
Test class distribution: {0: 593, 1: 408}
Class weights: tensor([1.1271, 0.8987])
Epoch 1/30, Loss: 0.5587, Val Accuracy: 0.8520
Epoch 2/30, Loss: 0.2604, Val Accuracy: 0.9371
Epoch 3/30, Loss: 0.1635, Val Accuracy: 0.9494
Epoch 4/30, Loss: 0.1596, Val Accuracy: 0.9593
Epoch 5/30, Loss: 0.1094, Val Accuracy: 0.9729
Epoch 6/30, Loss: 0.1047, Val Accuracy: 0.9827
Epoch 7/30, Loss: 0.0888, Val Accuracy: 0.9494
Epoch 8/30, Loss: 0.0802, Val Accuracy: 0.9852
Epoch 9/30, Loss: 0.0679, Val Accuracy: 0.9864
Epoch 10/30, Loss: 0.0688, Val Accuracy: 0.9889
Epoch 11/30, Loss: 0.0706, Val Accuracy: 0.9827
Epoch 12/30, Loss: 0.0560, Val Accuracy: 0.9901
Epoch 13/30, Loss: 0.0568, Val Accuracy: 0.9877
Epoch 14/30, Loss: 0.0518, Val Accuracy: 0.9840
Epoch 15/30, Loss: 0.0596, Val Accuracy: 0.9926
Epoch 16/30, Loss: 0.0588, Val Accuracy: 0.9914
Epoch 17/30, Loss: 0.0446, Val Accuracy: 0.9914
Epoch 18/30, Loss: 0.0536, Val Accuracy: 0.9926
Epoch 19/30, Loss:

In [2]:

torch.save(model, 'NewBestFCN.pth')